# 01 - Build the merged training set from three datasets

This notebook turns three sources into one classification set with 11 denominations
(0.1, 0.2, 0.5, 1, 2, 5, 10 DH coins and 20, 50, 100, 200 DH notes).

| source | what it is | how it is used |
|---|---|---|
| `web_moroccan_money` | 355 catalogue / web photos, one object per image, image-level labels | used as is, after removing duplicates |
| `moroccan_coins_yolo` | 184 phone photos with several coins each, polygon annotations (Roboflow, MIT) | one coin crop per annotated coin, cut the same way `detect.py` cuts them |
| `moroccan_currency_yolo` | 658 phone photos with one note or one 5 DH coin each (Roboflow, CC BY 4.0) | the four note classes as whole photos, the 5 DH photos are left out (see section 3) |

The most important job here is not the copying, it is the **groups**. Photos taken seconds apart, or copies of the same photo,
must never end up on both sides of a train/test split, otherwise the test score is inflated. Every image gets a group id
and the folds are built on groups.

In [ ]:
import sys, re, json, shutil, hashlib, datetime, collections, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import imagehash
from PIL import Image
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
import moneylib as ml

RAW, MERGED, FIG = ml.RAW, ml.MERGED, ml.RESULTS / "figures"
FIG.mkdir(parents=True, exist_ok=True)
if MERGED.exists():
    shutil.rmtree(MERGED)                       
MERGED.mkdir(parents=True)
rows = []                                       
print("root:", ROOT)

## 1. Web photos: duplicates and near-duplicates

Exact copies are removed with an md5 hash. Resized or re-cropped copies of the same note survive md5, so a 16x16 perceptual
hash (256 bits, also computed on the mirrored image) groups images of the same class that are closer than 40 bits.
The threshold is the one chosen by eye in the earlier cross-validation notebook: up to 40 bits the pairs are the same physical
note, from 44 bits they are different notes of the same design.

In [ ]:
HASH_THR = 40
web = []
for c in ml.CLASSES:
    for f in sorted((RAW / "web_moroccan_money" / c).iterdir()):
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".jfif"}:
            web.append({"src": f, "label": c, "md5": hashlib.md5(f.read_bytes()).hexdigest()})
web = pd.DataFrame(web)
n_raw = len(web)
web = web.drop_duplicates("md5").reset_index(drop=True)
print(f"web photos: {n_raw} -> {len(web)} after removing exact copies")

def hashes(p):
    im = Image.open(p).convert("RGB")
    a = imagehash.phash(im, hash_size=16).hash.flatten()
    b = imagehash.phash(im.transpose(Image.FLIP_LEFT_RIGHT), hash_size=16).hash.flatten()
    return a, b
hb = [hashes(p) for p in web.src]
A, B = np.array([h[0] for h in hb]), np.array([h[1] for h in hb])
dist = np.minimum((A[:, None] != A[None]).sum(2), (A[:, None] != B[None]).sum(2))
lab = web.label.to_numpy(dtype=str)
same = lab[:, None] == lab[None]
adj = (dist <= HASH_THR) & same
np.fill_diagonal(adj, False)
_, comp = connected_components(adj, directed=False)
web["group"] = ["web:%d" % g for g in comp]
cross = ((dist <= HASH_THR) & ~same).sum() // 2
print(f"near-duplicate groups: {len(set(comp))} groups for {len(web)} images | cross-class pairs under the threshold: {cross}")

for r in web.itertuples():
    dst = MERGED / "web" / r.label / r.src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(r.src, dst)
    rows.append({"path": f"web/{r.label}/{r.src.name}", "label": r.label, "source": "web", "group": r.group})

## 2. Phone coins: one crop per annotated coin

The Roboflow export contains every photo three times (augmented copies with a `.rf.<hash>` suffix in the file name), so
only one copy per source photo is kept. Each coin is cut with the same function `detect.py` uses to cut a detected coin
(`crop_circle`, 15% padding), so the classifier sees the same kind of crop at training time and at run time.

The dataset calls its classes `five, half, one, quarter, ten`. `quarter` is mapped to **0.2 DH** (the 20 centimes coin,
called "a quarter" in everyday Moroccan speech); the mapping is checked visually below. The group of a coin is its source photo.

In [ ]:
COIN_MAP = {"five": "5", "half": "0.5", "one": "1", "quarter": "0.2", "ten": "10"}
coin_names = ["five", "half", "one", "quarter", "ten"]          
assert re.search(r"names:\s*\['five', 'half', 'one', 'quarter', 'ten'\]", (RAW / "moroccan_coins_yolo" / "data.yaml").read_text())

def crop_circle(img, cx, cy, r, pad=1.15):                       
    half = int(r * pad) + 1
    p = cv2.copyMakeBorder(img, half, half, half, half, cv2.BORDER_REPLICATE)
    cx, cy = int(round(cx)) + half, int(round(cy)) + half
    return p[cy - half:cy + half, cx - half:cx + half]

rows = [r for r in rows if r["source"] != "phone_coin"]   # idempotent: re-running this cell must not duplicate coin rows
n_files = n_sources = 0
for split in ["train", "valid", "test"]:
    seen = set()
    for f in sorted((RAW / "moroccan_coins_yolo" / split / "images").glob("*.jpg")):
        n_files += 1
        source = f.name.split("_jpg.rf.")[0]
        if source in seen:
            continue
        seen.add(source)
        n_sources += 1
        img = cv2.imread(str(f)); H, W = img.shape[:2]
        for k, line in enumerate((RAW / "moroccan_coins_yolo" / split / "labels" / (f.stem + ".txt")).read_text().splitlines()):
            p = line.split()
            if len(p) < 5:
                continue
            label = COIN_MAP[coin_names[int(p[0])]]
            v = list(map(float, p[1:]))
            if len(v) > 4:                                       
                pts = np.array(v).reshape(-1, 2) * [W, H]
                x0, y0 = pts.min(0); x1, y1 = pts.max(0)
            else:                                                
                x, y, w, h = v
                x0, y0, x1, y1 = (x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H
            cx, cy, r = (x0 + x1) / 2, (y0 + y1) / 2, max(x1 - x0, y1 - y0) / 2
            if r < 8:
                continue
            out = MERGED / "phone_coin" / label / f"{source}__{k}.jpg"
            out.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(out), crop_circle(img, cx, cy, r))
            rows.append({"path": f"phone_coin/{label}/{out.name}", "label": label, "source": "phone_coin", "group": f"coin:{source}"})
print(f"{n_files} files = {n_sources} distinct photos (augmented copies dropped)")
print(pd.Series([r["label"] for r in rows if r["source"] == "phone_coin"]).value_counts().reindex(["0.2", "0.5", "1", "5", "10"]).to_string())

In [ ]:
fig, axes = plt.subplots(5, 10, figsize=(15, 7.8))
for r, c in enumerate(["0.2", "0.5", "1", "5", "10"]):
    ph = sorted((MERGED / "phone_coin" / c).glob("*.jpg"))[::max(1, len(list((MERGED / "phone_coin" / c).glob("*.jpg"))) // 5)][:5]
    wb = sorted((MERGED / "web" / c).glob("*"))[::max(1, len(list((MERGED / "web" / c).glob("*"))) // 5)][:5]
    for k, p in enumerate(ph + wb):
        axes[r, k].imshow(Image.open(p).convert("RGB")); axes[r, k].axis("off")
    axes[r, 0].set_title(f"{c} DH  (left: phone, right: web)", loc="left", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "01_coin_mapping_check.png", dpi=80); plt.show()

## 3. Phone currency photos: notes as whole photos, 5 DH photos left out

This dataset has one object per photo, but its boxes are useless for cropping: every box is the same 0.9 x 0.9 rectangle
centred on the image (81% of the area), so the label is really an image-level label.
Cutting the notes out with the project's own detectors was tried and it failed on these photos: with the colour/shape rules only
30 of 658 photos gave exactly one correct object, and FastSAM mostly returned parts of the keyboard, the desk or the
printed folder the notes lie on. A crop that is wrong teaches the classifier the background.

Decision:
- **20, 50, 100, 200 DH notes:** the whole photo is used (source `phone_note`). The note fills a large part of the frame and
  the label is correct, so this is clean supervision, and `detect.py` already classifies the whole picture when a note fills the frame.
- **5 DH coins (156 photos):** left out. The coin is a few percent of the frame, so a whole-photo label would mostly describe
  the carpet or the table, and the automatic crops were not coins (checked on a sample). The 5 DH class is covered by the phone coins of section 2 and the web photos.

Photos are named with a time stamp, and many are taken a few seconds apart of the same note. Photos of the same class less than
10 seconds apart are chained into one **burst group**.

In [ ]:
cur_names = ["100dh", "200dh", "20dh", "50dh", "5dh"]            # order in data.yaml
assert re.search(r"names:\s*\['100dh', '200dh', '20dh', '50dh', '5dh'\]", (RAW / "moroccan_currency_yolo" / "data.yaml").read_text())
cur = []
for split in ["train", "valid", "test"]:
    for f in sorted((RAW / "moroccan_currency_yolo" / split / "images").glob("*.jpg")):
        name = cur_names[int((RAW / "moroccan_currency_yolo" / split / "labels" / (f.stem + ".txt")).read_text().split()[0])]
        m = re.match(r"(\d{8})_(\d{6})", f.name)
        ts = datetime.datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S").timestamp()
        cur.append({"src": f, "cls": name, "ts": ts})
cur = pd.DataFrame(cur)
print("currency photos:", len(cur), "|", cur.cls.value_counts().to_dict())
kept = cur[cur.cls != "5dh"].sort_values(["cls", "ts"]).reset_index(drop=True)
burst, last, cls_prev = [], None, None
for r in kept.itertuples():
    if cls_prev != r.cls or r.ts - last > 10:
        burst.append(len(set(burst)))
    else:
        burst.append(burst[-1])
    cls_prev, last = r.cls, r.ts
kept["burst"] = burst
print(f"kept {len(kept)} note photos in {kept.burst.nunique()} burst groups | left out: {int((cur.cls == '5dh').sum())} photos of 5 DH")
VAL = {"100dh": "100", "200dh": "200", "20dh": "20", "50dh": "50"}
for r in kept.itertuples():
    label = VAL[r.cls]
    dst = MERGED / "phone_note" / label / r.src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(r.src, dst)
    rows.append({"path": f"phone_note/{label}/{r.src.name}", "label": label, "source": "phone_note", "group": f"burst:{r.burst}"})

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(15, 7.8))
for r, c in enumerate(["20", "50", "100", "200"]):
    fs = sorted((MERGED / "phone_note" / c).glob("*.jpg")); fs = fs[::max(1, len(fs) // 8)][:8]
    for k, p in enumerate(fs):
        axes[r, k].imshow(Image.open(p).convert("RGB")); axes[r, k].axis("off")
    axes[r, 0].set_title(f"{c} DH (whole phone photos)", loc="left", fontsize=9)
plt.tight_layout(); plt.savefig(FIG / "01_phone_notes_sample.png", dpi=80); plt.show()

## 4. Index and grouped folds

`StratifiedGroupKFold` keeps class proportions and never splits a group. Fold 0 is the test fold used by the next notebooks
(about 20% of the images), and it is never used for training or for choosing anything. The assertion fails loudly if a group appears on both sides.

In [ ]:
df = pd.DataFrame(rows)
assert df.path.is_unique, "duplicate rows in the index: re-run the notebook from the first cell"
df["y"] = df.label.map({c: i for i, c in enumerate(ml.CLASSES)})
sgkf = StratifiedGroupKFold(5, shuffle=True, random_state=ml.SEED)
df["fold"] = -1
for k, (tr, te) in enumerate(sgkf.split(np.zeros(len(df)), df.y, df.group)):
    assert not set(df.group.iloc[tr]) & set(df.group.iloc[te]), "a group straddles train and test"
    df.loc[df.index[te], "fold"] = k
df[["path", "label", "source", "group", "fold"]].to_csv(MERGED / "index.csv", index=False)

print(f"{len(df)} images, {df.group.nunique()} groups")
tab = pd.crosstab(df.label, df.source).reindex(ml.CLASSES)
tab["total"] = tab.sum(axis=1)
tab["test (fold 0)"] = df[df.fold == 0].groupby("label").size().reindex(ml.CLASSES).fillna(0).astype(int)
display(tab.T)
assert (tab["test (fold 0)"] > 0).all(), "every class needs at least one test image"

In [ ]:
tab[["web", "phone_coin", "phone_note"]].plot(kind="bar", stacked=True, figsize=(9, 3.2), color=["#4C78A8", "#F58518", "#54A24B"])
plt.ylabel("images"); plt.xlabel("denomination (DH)"); plt.title("Merged training set by class and source")
plt.tight_layout(); plt.savefig(FIG / "01_class_distribution.png", dpi=80); plt.show()

## What this set can and cannot say

- Coins have real phone photos for 0.2, 0.5, 1, 5 and 10 DH. **0.1 DH and 2 DH have web photos only**, and 2 DH is the smallest class.
- Notes have real phone photos for all four values, but as whole frames, not as tight crops.
- Nothing here contains a "not money" class, so the model must always pick one of the 11 values; the confidence threshold in `detect.py` is the only guard against objects that are not money.
- The phone photos come from a few people and a few cameras. Scores on them say how the model handles that kind of photo, not every camera.

Next: `02_train_and_evaluate.ipynb`.